# Práctica SQL 02 — Caso 02: Academia Online (preguntas)

10 preguntas básicas y 20 intermedias.

**Plataforma:** Databricks SQL · Delta Lake · Unity Catalog

## Descripción

Una plataforma latinoamericana ofrece cursos de tecnología, negocios, marketing y diseño. La empresa desea analizar estudiantes, instructores, inscripciones, pagos, sesiones de estudio y reseñas para mejorar el aprendizaje y los ingresos.

### Reglas de negocio para los ejercicios

- Considera como ingreso únicamente los pagos con `estado_pago = 'aprobado'`.
- Una inscripción completada tiene `estado = 'completado'`.
- La tasa de finalización se calcula como inscripciones completadas entre inscripciones totales.

## Tablas

| Tabla | Columnas y tipos | Descripción |
|---|---|---|
| `estudiantes` | `estudiante_id STRING`; `nombre_estudiante STRING`; `pais STRING`; `ciudad STRING`; `plan STRING`; `fecha_registro DATE`; `edad INT` | País, plan y fecha de registro. |
| `instructores` | `instructor_id STRING`; `nombre_instructor STRING`; `pais STRING`; `especialidad STRING`; `fecha_ingreso DATE` | Especialidad y fecha de ingreso. |
| `cursos` | `curso_id STRING`; `instructor_id STRING`; `titulo_curso STRING`; `categoria STRING`; `nivel STRING`; `precio DECIMAL(10,2)`; `duracion_horas DECIMAL(8,2)`; `fecha_publicacion DATE`; `activo BOOLEAN` | Categoría, nivel, precio, duración e instructor. |
| `inscripciones` | `inscripcion_id STRING`; `estudiante_id STRING`; `curso_id STRING`; `fecha_inscripcion DATE`; `estado STRING`; `porcentaje_avance INT`; `nota_final INT` *(admite NULL)* | Relación estudiante-curso, avance y nota. |
| `pagos` | `pago_id STRING`; `inscripcion_id STRING`; `fecha_pago DATE`; `monto DECIMAL(10,2)`; `metodo_pago STRING`; `estado_pago STRING` | Monto, método y estado del pago. |
| `sesiones` | `sesion_id STRING`; `inscripcion_id STRING`; `fecha_sesion DATE`; `minutos_vistos INT`; `dispositivo STRING` | Actividad de estudio por inscripción. |
| `resenas` | `resena_id STRING`; `inscripcion_id STRING`; `fecha_resena DATE`; `calificacion INT`; `comentario STRING` | Calificación y comentario de cursos completados. |

## Relaciones

- `instructores.instructor_id` → `cursos.instructor_id` (1 a N)
- `estudiantes.estudiante_id` → `inscripciones.estudiante_id` (1 a N)
- `cursos.curso_id` → `inscripciones.curso_id` (1 a N)
- `inscripciones.inscripcion_id` → `pagos.inscripcion_id` (1 a N)
- `inscripciones.inscripcion_id` → `sesiones.inscripcion_id` (1 a N)
- `inscripciones.inscripcion_id` → `resenas.inscripcion_id` (1 a N)

## Consideraciones

- Los archivos TXT usan formato CSV con encabezado y codificación UTF-8.
- Las fechas usan el formato `YYYY-MM-DD`.
- `nota_final` solo se completa cuando la inscripción está terminada (en otro caso es `NULL`).
- Los cursos gratuitos tienen pagos aprobados por monto `0.00` para conservar la trazabilidad.

---
## PARTE 0 — Preparación del entorno (DDL y carga)

> Si ya creaste y cargaste las tablas de este caso, salta directamente a la **PARTE 1**.

### 0.1 Esquemas y volumen

| Objeto | Nombre | Para qué sirve |
|---|---|---|
| Esquema | `workspace.sql_practice_raw` | Contiene el volumen con los archivos originales. |
| Volumen | `workspace.sql_practice_raw.datasets` | Almacena los `.txt` de ambos casos, cada uno en su carpeta. |
| Esquema | `workspace.sql_practice_academia` | Contiene las 7 tablas de este caso. |

En Databricks Free Edition el catálogo por defecto es `workspace`. Si el tuyo se llama `main` u otro nombre, reemplaza `workspace` en todo el notebook, incluidas las rutas `/Volumes/workspace/...`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_raw;
CREATE VOLUME IF NOT EXISTS workspace.sql_practice_raw.datasets;

CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_academia;

### 0.2 Subir los archivos al volumen

1. En **Catalog Explorer** abre el volumen `workspace` → `sql_practice_raw` → **Volumes** → `datasets`.
2. Crea la carpeta **`caso_02_academia_online`** (si aún no existe).
3. Dentro de esa carpeta, sube los 7 archivos de `datos_txt/`: `estudiantes.txt`, `instructores.txt`, `cursos.txt`, `inscripciones.txt`, `pagos.txt`, `sesiones.txt`, `resenas.txt`.
4. Ejecuta la siguiente celda: deben aparecer los 7 archivos.

In [0]:
%sql
LIST '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/';

path,name,size,modification_time
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/cursos.txt,cursos.txt,3809,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/estudiantes.txt,estudiantes.txt,10879,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/inscripciones.txt,inscripciones.txt,24318,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/instructores.txt,instructores.txt,1090,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/pagos.txt,pagos.txt,26029,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/resenas.txt,resenas.txt,6358,1792247275000
/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/sesiones.txt,sesiones.txt,161056,1792247275000


### 0.3 Eliminar tablas previas

Permite volver a ejecutar la preparación desde cero. Las tablas se eliminan en **orden inverso a sus dependencias**: primero las hijas y luego las padres.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_academia;

DROP TABLE IF EXISTS resenas;
DROP TABLE IF EXISTS sesiones;
DROP TABLE IF EXISTS pagos;
DROP TABLE IF EXISTS inscripciones;
DROP TABLE IF EXISTS cursos;
DROP TABLE IF EXISTS instructores;
DROP TABLE IF EXISTS estudiantes;

### 0.4 Crear las tablas (DDL)

- Todas las columnas son `NOT NULL`, salvo las indicadas como *admite NULL*. Delta Lake **sí** valida esta restricción al insertar.
- `PRIMARY KEY` y `FOREIGN KEY` en Unity Catalog son restricciones **informativas**: documentan el modelo, pero **no se validan** al insertar. Por eso en el paso 0.6 verificamos la integridad con consultas.
- Las tablas se crean de **padres a hijas**: una clave foránea solo puede apuntar a una tabla que ya tenga su clave primaria.

#### `estudiantes`

In [0]:
%sql
--CREATE OR REPLACE -> EQUIVALENTE A DROPEAR Y VOLVER A CREAR
CREATE TABLE estudiantes (
  estudiante_id      STRING  NOT NULL COMMENT 'Identificador del estudiante',
  nombre_estudiante  STRING  NOT NULL COMMENT 'Nombre completo',
  pais               STRING  NOT NULL COMMENT 'País de residencia',
  ciudad             STRING  NOT NULL COMMENT 'Ciudad de residencia',
  plan               STRING  NOT NULL COMMENT 'Plan de suscripción',
  fecha_registro     DATE    NOT NULL COMMENT 'Fecha de registro',
  edad               INT     NOT NULL COMMENT 'Edad en años',
  CONSTRAINT pk_estudiantes PRIMARY KEY (estudiante_id)
)
COMMENT 'Estudiantes de la plataforma.';

#### `instructores`

In [0]:
%sql
CREATE TABLE instructores (
  instructor_id      STRING  NOT NULL COMMENT 'Identificador del instructor',
  nombre_instructor  STRING  NOT NULL COMMENT 'Nombre completo',
  pais               STRING  NOT NULL COMMENT 'País de residencia',
  especialidad       STRING  NOT NULL COMMENT 'Especialidad del instructor',
  fecha_ingreso      DATE    NOT NULL COMMENT 'Fecha de ingreso a la plataforma',
  CONSTRAINT pk_instructores PRIMARY KEY (instructor_id)
)
COMMENT 'Instructores y su especialidad.';

#### `cursos`

In [0]:
%sql
CREATE TABLE cursos (
  curso_id           STRING         NOT NULL COMMENT 'Identificador del curso',
  instructor_id      STRING         NOT NULL COMMENT 'Instructor a cargo',
  titulo_curso       STRING         NOT NULL COMMENT 'Título del curso',
  categoria          STRING         NOT NULL COMMENT 'Categoría del curso',
  nivel              STRING         NOT NULL COMMENT 'Nivel: Básico, Intermedio o Avanzado',
  precio             DECIMAL(10,2)  NOT NULL COMMENT 'Precio en dólares (0 = gratuito)',
  duracion_horas     DECIMAL(8,2)   NOT NULL COMMENT 'Duración del contenido en horas',
  fecha_publicacion  DATE           NOT NULL COMMENT 'Fecha de publicación',
  activo             BOOLEAN        NOT NULL COMMENT 'Indica si el curso está activo',
  CONSTRAINT pk_cursos PRIMARY KEY (curso_id),
  CONSTRAINT fk_cursos_instructores FOREIGN KEY (instructor_id) REFERENCES instructores (instructor_id)
)
COMMENT 'Catálogo de cursos.';

#### `inscripciones`

In [0]:
%sql
CREATE TABLE inscripciones (
  inscripcion_id     STRING  NOT NULL COMMENT 'Identificador de la inscripción',
  estudiante_id      STRING  NOT NULL COMMENT 'Estudiante inscrito',
  curso_id           STRING  NOT NULL COMMENT 'Curso inscrito',
  fecha_inscripcion  DATE    NOT NULL COMMENT 'Fecha de inscripción',
  estado             STRING  NOT NULL COMMENT 'Estado de la inscripción',
  porcentaje_avance  INT     NOT NULL COMMENT 'Avance del curso (0 a 100)',
  nota_final         INT              COMMENT 'Nota final (NULL si no ha terminado)',
  CONSTRAINT pk_inscripciones PRIMARY KEY (inscripcion_id),
  CONSTRAINT fk_inscripciones_estudiantes FOREIGN KEY (estudiante_id) REFERENCES estudiantes (estudiante_id),
  CONSTRAINT fk_inscripciones_cursos FOREIGN KEY (curso_id) REFERENCES cursos (curso_id)
)
COMMENT 'Relación estudiante-curso.';

#### `pagos`

In [0]:
%sql
CREATE TABLE pagos (
  pago_id         STRING         NOT NULL COMMENT 'Identificador del pago',
  inscripcion_id  STRING         NOT NULL COMMENT 'Inscripción pagada',
  fecha_pago      DATE           NOT NULL COMMENT 'Fecha del pago',
  monto           DECIMAL(10,2)  NOT NULL COMMENT 'Monto pagado',
  metodo_pago     STRING         NOT NULL COMMENT 'Método de pago',
  estado_pago     STRING         NOT NULL COMMENT 'Estado del pago',
  CONSTRAINT pk_pagos PRIMARY KEY (pago_id),
  CONSTRAINT fk_pagos_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Pagos de cada inscripción.';

#### `sesiones`

In [0]:
%sql
CREATE TABLE sesiones (
  sesion_id       STRING  NOT NULL COMMENT 'Identificador de la sesión',
  inscripcion_id  STRING  NOT NULL COMMENT 'Inscripción a la que pertenece',
  fecha_sesion    DATE    NOT NULL COMMENT 'Fecha de la sesión',
  minutos_vistos  INT     NOT NULL COMMENT 'Minutos de contenido vistos',
  dispositivo     STRING  NOT NULL COMMENT 'Dispositivo utilizado',
  CONSTRAINT pk_sesiones PRIMARY KEY (sesion_id),
  CONSTRAINT fk_sesiones_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Sesiones de estudio por inscripción.';

#### `resenas`

In [0]:
%sql
CREATE TABLE resenas (
  resena_id       STRING  NOT NULL COMMENT 'Identificador de la reseña',
  inscripcion_id  STRING  NOT NULL COMMENT 'Inscripción reseñada',
  fecha_resena    DATE    NOT NULL COMMENT 'Fecha de la reseña',
  calificacion    INT     NOT NULL COMMENT 'Calificación de 1 a 5',
  comentario      STRING  NOT NULL COMMENT 'Comentario del estudiante',
  CONSTRAINT pk_resenas PRIMARY KEY (resena_id),
  CONSTRAINT fk_resenas_inscripciones FOREIGN KEY (inscripcion_id) REFERENCES inscripciones (inscripcion_id)
)
COMMENT 'Reseñas de cursos completados.';

### 0.5 Cargar los datos

Usamos `read_files` para leer cada `.txt` del volumen:

- `format => 'csv'` y `header => true`: texto delimitado por comas con encabezado.
- `schema => '...'`: define el tipo de cada columna en lugar de dejar que Databricks lo infiera. Los campos vacíos se cargan como `NULL`.
- `mode => 'FAILFAST'`: si alguna fila no se puede convertir al tipo indicado, la carga falla en lugar de dejar datos incorrectos.

#### `estudiantes`

In [0]:
%sql
INSERT INTO estudiantes
SELECT
  estudiante_id,
  nombre_estudiante,
  pais,
  ciudad,
  plan,
  fecha_registro,
  edad
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/estudiantes.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'estudiante_id STRING, nombre_estudiante STRING, pais STRING, ciudad STRING, plan STRING, fecha_registro DATE, edad INT'
);

SELECT * FROM estudiantes LIMIT 5;

estudiante_id,nombre_estudiante,pais,ciudad,plan,fecha_registro,edad
E0001,Natalia Reyes,Perú,Arequipa,Gratis,2024-05-20,23
E0002,Marco Peña,Colombia,Bogotá,Básico,2025-09-07,37
E0003,Valeria López,Perú,Cusco,Gratis,2025-11-23,24
E0004,Fernando Reyes,México,Querétaro,Básico,2024-07-02,52
E0005,Andrea Chávez,Colombia,Bogotá,Gratis,2024-01-15,32


#### `instructores`

In [0]:
%sql
INSERT INTO instructores
SELECT
  instructor_id,
  nombre_instructor,
  pais,
  especialidad,
  fecha_ingreso
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/instructores.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'instructor_id STRING, nombre_instructor STRING, pais STRING, especialidad STRING, fecha_ingreso DATE'
);

SELECT * FROM instructores LIMIT 5;

instructor_id,nombre_instructor,pais,especialidad,fecha_ingreso
I001,Lucía Paredes,Ecuador,Data Analytics,2025-01-31
I002,Luis Herrera,Colombia,Programación,2021-08-30
I003,Paola Espinoza,México,Marketing,2023-04-13
I004,Hugo Peña,México,Diseño UX,2021-09-29
I005,Luis Cárdenas,Ecuador,Negocios,2021-06-01


#### `cursos`

In [0]:
%sql
INSERT INTO cursos
SELECT
  curso_id,
  instructor_id,
  titulo_curso,
  categoria,
  nivel,
  precio,
  duracion_horas,
  fecha_publicacion,
  activo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/cursos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'curso_id STRING, instructor_id STRING, titulo_curso STRING, categoria STRING, nivel STRING, precio DECIMAL(10,2), duracion_horas DECIMAL(8,2), fecha_publicacion DATE, activo BOOLEAN'
);

SELECT * FROM cursos LIMIT 5;

curso_id,instructor_id,titulo_curso,categoria,nivel,precio,duracion_horas,fecha_publicacion,activo
K0001,I015,SQL desde Cero,Data Analytics,Avanzado,55.00,13.32,2025-04-05,true
K0002,I008,Power BI Aplicado,Data Analytics,Básico,0.00,14.53,2026-01-17,true
K0003,I008,Excel para Analistas,Data Analytics,Intermedio,75.00,6.88,2025-09-26,true
K0004,I008,Storytelling con Datos,Data Analytics,Intermedio,65.00,11.41,2024-03-10,true
K0005,I015,Estadística Práctica,Data Analytics,Avanzado,90.00,9.73,2024-10-27,true


#### `inscripciones`

In [0]:
%sql
INSERT INTO inscripciones
SELECT
  inscripcion_id,
  estudiante_id,
  curso_id,
  fecha_inscripcion,
  estado,
  porcentaje_avance,
  nota_final
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/inscripciones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'inscripcion_id STRING, estudiante_id STRING, curso_id STRING, fecha_inscripcion DATE, estado STRING, porcentaje_avance INT, nota_final INT'
);

SELECT * FROM inscripciones LIMIT 5;

inscripcion_id,estudiante_id,curso_id,fecha_inscripcion,estado,porcentaje_avance,nota_final
N00001,E0010,K0041,2025-12-21,en_curso,35,null
N00002,E0114,K0037,2025-10-15,completado,100,80
N00003,E0177,K0018,2025-05-09,en_curso,41,null
N00004,E0013,K0003,2026-06-11,en_curso,56,null
N00005,E0072,K0008,2026-03-08,completado,100,85


#### `pagos`

In [0]:
%sql
INSERT INTO pagos
SELECT
  pago_id,
  inscripcion_id,
  fecha_pago,
  monto,
  metodo_pago,
  estado_pago
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/pagos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pago_id STRING, inscripcion_id STRING, fecha_pago DATE, monto DECIMAL(10,2), metodo_pago STRING, estado_pago STRING'
);

SELECT * FROM pagos LIMIT 5;

pago_id,inscripcion_id,fecha_pago,monto,metodo_pago,estado_pago
G00001,N00001,2025-12-21,90.25,Tarjeta,reembolsado
G00002,N00002,2025-10-15,104.50,Mercado Pago,aprobado
G00003,N00003,2025-05-09,0.00,Transferencia,aprobado
G00004,N00004,2026-06-11,71.25,PayPal,reembolsado
G00005,N00005,2026-03-08,40.00,Transferencia,aprobado


#### `sesiones`

In [0]:
%sql
INSERT INTO sesiones
SELECT
  sesion_id,
  inscripcion_id,
  fecha_sesion,
  minutos_vistos,
  dispositivo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/sesiones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'sesion_id STRING, inscripcion_id STRING, fecha_sesion DATE, minutos_vistos INT, dispositivo STRING'
);

SELECT * FROM sesiones LIMIT 5;

sesion_id,inscripcion_id,fecha_sesion,minutos_vistos,dispositivo
S000001,N00001,2025-12-24,51,Móvil
S000002,N00001,2025-12-31,87,Móvil
S000003,N00001,2026-01-18,107,Móvil
S000004,N00001,2026-01-28,28,Móvil
S000005,N00001,2026-02-07,53,Tablet


#### `resenas`

In [0]:
%sql
INSERT INTO resenas
SELECT
  resena_id,
  inscripcion_id,
  fecha_resena,
  calificacion,
  comentario
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_02_academia_online/resenas.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'resena_id STRING, inscripcion_id STRING, fecha_resena DATE, calificacion INT, comentario STRING'
);

SELECT * FROM resenas LIMIT 5;

resena_id,inscripcion_id,fecha_resena,calificacion,comentario
Q00001,N00002,2025-12-01,4,Buen contenido
Q00002,N00007,2026-01-02,3,Podría tener más ejemplos
Q00003,N00010,2025-01-09,5,Excelente curso
Q00004,N00012,2026-07-20,5,Superó mis expectativas
Q00005,N00013,2026-02-01,4,Recomendado


### 0.6 Validación

Todas las tablas deben mostrar `estado = OK`.

In [0]:
%sql
WITH conteos AS (
  SELECT 'estudiantes' AS tabla, COUNT(*) AS filas, 180 AS filas_esperadas FROM estudiantes
  UNION ALL
  SELECT 'instructores' AS tabla, COUNT(*) AS filas, 20 AS filas_esperadas FROM instructores
  UNION ALL
  SELECT 'cursos' AS tabla, COUNT(*) AS filas, 45 AS filas_esperadas FROM cursos
  UNION ALL
  SELECT 'inscripciones' AS tabla, COUNT(*) AS filas, 520 AS filas_esperadas FROM inscripciones
  UNION ALL
  SELECT 'pagos' AS tabla, COUNT(*) AS filas, 520 AS filas_esperadas FROM pagos
  UNION ALL
  SELECT 'sesiones' AS tabla, COUNT(*) AS filas, 4329 AS filas_esperadas FROM sesiones
  UNION ALL
  SELECT 'resenas' AS tabla, COUNT(*) AS filas, 135 AS filas_esperadas FROM resenas
)
SELECT
  tabla,
  filas,
  filas_esperadas,
  IF(filas = filas_esperadas, 'OK', 'REVISAR') AS estado
FROM conteos;

tabla,filas,filas_esperadas,estado
estudiantes,180,180,OK
instructores,20,20,OK
cursos,45,45,OK
inscripciones,520,520,OK
pagos,520,520,OK
sesiones,4329,4329,OK
resenas,135,135,OK


Integridad referencial: como las claves foráneas son informativas, comprobamos que no existan filas hijas que apunten a un padre inexistente. Todas las relaciones deben mostrar `huerfanos = 0`.

In [0]:
%sql
SELECT 'cursos.instructor_id -> instructores' AS relacion, COUNT(*) AS huerfanos
FROM cursos AS h
LEFT JOIN instructores AS p ON h.instructor_id = p.instructor_id
WHERE p.instructor_id IS NULL
UNION ALL
SELECT 'inscripciones.estudiante_id -> estudiantes' AS relacion, COUNT(*) AS huerfanos
FROM inscripciones AS h
LEFT JOIN estudiantes AS p ON h.estudiante_id = p.estudiante_id
WHERE p.estudiante_id IS NULL
UNION ALL
SELECT 'inscripciones.curso_id -> cursos' AS relacion, COUNT(*) AS huerfanos
FROM inscripciones AS h
LEFT JOIN cursos AS p ON h.curso_id = p.curso_id
WHERE p.curso_id IS NULL
UNION ALL
SELECT 'pagos.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM pagos AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL
UNION ALL
SELECT 'sesiones.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM sesiones AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL
UNION ALL
SELECT 'resenas.inscripcion_id -> inscripciones' AS relacion, COUNT(*) AS huerfanos
FROM resenas AS h
LEFT JOIN inscripciones AS p ON h.inscripcion_id = p.inscripcion_id
WHERE p.inscripcion_id IS NULL;

relacion,huerfanos
cursos.instructor_id -> instructores,0
inscripciones.estudiante_id -> estudiantes,0
inscripciones.curso_id -> cursos,0
pagos.inscripcion_id -> inscripciones,0
sesiones.inscripcion_id -> inscripciones,0
resenas.inscripcion_id -> inscripciones,0


---
## Configuración para resolver las preguntas

Ejecuta esta celda cada vez que abras el notebook: las consultas usan nombres de tablas sin catálogo ni esquema.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_academia;

SHOW TABLES;

database,tableName,isTemporary
sql_practice_academia,cursos,false
sql_practice_academia,estudiantes,false
sql_practice_academia,inscripciones,false
sql_practice_academia,instructores,false
sql_practice_academia,pagos,false
sql_practice_academia,resenas,false
sql_practice_academia,sesiones,false


---
## PARTE 1 — Preguntas básicas (1-10)

### 1. Todos los estudiantes

Muestra todos los registros de `estudiantes`.

In [0]:
%sql
SELECT * FROM estudiantes;

estudiante_id,nombre_estudiante,pais,ciudad,plan,fecha_registro,edad
E0001,Natalia Reyes,Perú,Arequipa,Gratis,2024-05-20,23
E0002,Marco Peña,Colombia,Bogotá,Básico,2025-09-07,37
E0003,Valeria López,Perú,Cusco,Gratis,2025-11-23,24
E0004,Fernando Reyes,México,Querétaro,Básico,2024-07-02,52
E0005,Andrea Chávez,Colombia,Bogotá,Gratis,2024-01-15,32
E0006,Álvaro Acosta,Chile,Concepción,Básico,2024-06-22,48
E0007,Gabriela Vargas,Ecuador,Guayaquil,Gratis,2026-05-20,46
E0008,Gabriela Rojas,Perú,Lima,Básico,2025-10-02,26
E0009,Fernando Valdez,Argentina,Córdoba,Premium,2025-03-29,37
E0010,Gabriela Aguilar,Ecuador,Quito,Básico,2025-03-08,35


### 2. Cursos activos

Lista el ID, título, categoría y precio de los cursos activos.

In [0]:
%sql
SELECT curso_id, titulo_curso, categoria, precio 
FROM cursos
WHERE activo = TRUE;

curso_id,titulo_curso,categoria,precio
K0001,SQL desde Cero,Data Analytics,55.00
K0002,Power BI Aplicado,Data Analytics,0.00
K0003,Excel para Analistas,Data Analytics,75.00
K0004,Storytelling con Datos,Data Analytics,65.00
K0005,Estadística Práctica,Data Analytics,90.00
K0006,Databricks Fundamentals,Data Analytics,35.00
K0007,Modelado Dimensional,Data Analytics,65.00
K0008,Python Esencial,Programación,40.00
K0009,JavaScript Moderno,Programación,45.00
K0010,APIs con FastAPI,Programación,55.00


### 3. Categorías únicas

Obtén las categorías únicas de cursos, ordenadas alfabéticamente.

In [0]:
%sql
SELECT DISTINCT categoria 
FROM cursos
ORDER BY categoria;

curso_id,instructor_id,titulo_curso,categoria,nivel,precio,duracion_horas,fecha_publicacion,activo
K0001,I015,SQL desde Cero,Data Analytics,Avanzado,55.00,13.32,2025-04-05,true
K0002,I008,Power BI Aplicado,Data Analytics,Básico,0.00,14.53,2026-01-17,true
K0003,I008,Excel para Analistas,Data Analytics,Intermedio,75.00,6.88,2025-09-26,true
K0004,I008,Storytelling con Datos,Data Analytics,Intermedio,65.00,11.41,2024-03-10,true
K0005,I015,Estadística Práctica,Data Analytics,Avanzado,90.00,9.73,2024-10-27,true
K0006,I008,Databricks Fundamentals,Data Analytics,Básico,35.00,14.08,2023-04-01,true
K0007,I015,Modelado Dimensional,Data Analytics,Intermedio,65.00,9.22,2023-01-22,true
K0008,I009,Python Esencial,Programación,Intermedio,40.00,6.49,2025-11-26,true
K0009,I016,JavaScript Moderno,Programación,Básico,45.00,25.44,2023-10-08,true
K0010,I016,APIs con FastAPI,Programación,Intermedio,55.00,4.32,2024-11-03,true


### 4. Cursos básicos ordenados por precio

Muestra los cursos de nivel `Básico`, ordenados por precio de menor a mayor.

In [0]:
%sql
SELECT *
FROM cursos
WHERE nivel ="Básico"
ORDER BY precio;

curso_id,instructor_id,titulo_curso,categoria,nivel,precio,duracion_horas,fecha_publicacion,activo
K0002,I008,Power BI Aplicado,Data Analytics,Básico,0.00,14.53,2026-01-17,true
K0043,I007,Visión Computacional,Inteligencia Artificial,Básico,0.00,4.90,2025-06-17,true
K0018,I017,Analítica de Campañas,Marketing,Básico,0.00,29.55,2024-03-11,true
K0017,I003,Contenido para Redes,Marketing,Básico,0.00,26.12,2024-06-30,true
K0031,I005,Liderazgo de Equipos,Negocios,Básico,25.00,20.25,2023-01-02,true
K0035,I020,Azure para Principiantes,Cloud,Básico,30.00,21.21,2023-12-08,false
K0023,I004,Research con Usuarios,Diseño UX,Básico,30.00,17.79,2025-04-06,true
K0021,I018,Fundamentos de UX,Diseño UX,Básico,30.00,9.99,2023-05-05,true
K0034,I013,Google Cloud Essentials,Cloud,Básico,30.00,28.89,2023-02-19,true
K0006,I008,Databricks Fundamentals,Data Analytics,Básico,35.00,14.08,2023-04-01,true


### 5. Cantidad de instructores

Cuenta cuántos instructores existen.

In [0]:
%sql
SELECT COUNT(1) AS total_instructores
FROM instructores;

total_instructores
20


### 6. Duración promedio de cursos

Calcula la duración promedio de los cursos.

In [0]:
%sql
SELECT AVG(duracion_horas) AS duracion_promedio_horas
FROM cursos;

duracion_promedio_horas
20.374889


### 7. Inscripciones completadas

Muestra todas las inscripciones completadas.

In [0]:
%sql
SELECT * FROM inscripciones
WHERE estado = 'completado';

inscripcion_id,estudiante_id,curso_id,fecha_inscripcion,estado,porcentaje_avance,nota_final
N00002,E0114,K0037,2025-10-15,completado,100,80
N00005,E0072,K0008,2026-03-08,completado,100,85
N00007,E0075,K0003,2025-10-24,completado,100,70
N00008,E0032,K0024,2025-08-04,completado,100,82
N00010,E0093,K0014,2024-12-17,completado,100,100
N00011,E0115,K0037,2026-04-20,completado,100,92
N00012,E0059,K0023,2026-04-29,completado,100,95
N00013,E0014,K0012,2026-01-12,completado,100,78
N00014,E0093,K0035,2026-04-24,completado,100,65
N00015,E0160,K0029,2025-12-20,completado,100,85


### 8. Pagos aprobados mayores a 50

Lista los pagos aprobados cuyo monto sea mayor a 50.

In [0]:
%sql
SELECT *
FROM pagos
WHERE estado_pago = 'aprobado' AND monto > 50
ORDER BY monto;

pago_id,inscripcion_id,fecha_pago,monto,metodo_pago,estado_pago
G00078,N00078,2026-05-18,52.25,Transferencia,aprobado
G00058,N00058,2026-04-29,52.25,Mercado Pago,aprobado
G00036,N00036,2026-06-01,52.25,Mercado Pago,aprobado
G00439,N00439,2026-02-12,52.25,Tarjeta,aprobado
G00033,N00033,2026-05-05,52.25,Tarjeta,aprobado
G00091,N00091,2026-06-24,52.25,Tarjeta,aprobado
G00185,N00185,2025-12-12,52.25,Tarjeta,aprobado
G00118,N00118,2025-07-29,52.25,Tarjeta,aprobado
G00183,N00183,2026-04-03,52.25,Tarjeta,aprobado
G00410,N00410,2025-11-22,52.25,Yape,aprobado


### 9. Sesiones desde móvil

Muestra las sesiones realizadas desde un dispositivo `Móvil`.

In [0]:
%sql
SELECT *
FROM sesiones
WHERE dispositivo = 'Móvil';

sesion_id,inscripcion_id,fecha_sesion,minutos_vistos,dispositivo
S000001,N00001,2025-12-24,51,Móvil
S000002,N00001,2025-12-31,87,Móvil
S000003,N00001,2026-01-18,107,Móvil
S000004,N00001,2026-01-28,28,Móvil
S000008,N00001,2026-05-14,109,Móvil
S000012,N00002,2025-12-16,78,Móvil
S000018,N00003,2025-05-25,77,Móvil
S000019,N00003,2025-06-03,85,Móvil
S000023,N00003,2025-09-29,84,Móvil
S000024,N00004,2026-06-13,119,Móvil


### 10. Cantidad de reseñas por calificación

Cuenta cuántas reseñas existen por cada calificación presente en los datos.

In [0]:
%sql
SELECT 
    calificacion,
    COUNT(1) AS cantidad_resenas
FROM resenas
GROUP BY calificacion
ORDER BY calificacion;

calificacion,cantidad_resenas
2,6
3,28
4,57
5,44


---
## PARTE 2 — Preguntas intermedias (11-30)

### 11. Cursos con cantidad de inscripciones, incluyendo cursos sin inscripciones

Muestra cada curso con su cantidad total de inscripciones, incluyendo cursos sin inscripciones.

In [0]:
%sql
SELECT 
    c.curso_id,
    c.titulo_curso,
    COUNT(1) AS cantidad_inscripciones
FROM cursos AS c
LEFT JOIN inscripciones AS i
ON c.curso_id = i.curso_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY cantidad_inscripciones DESC;

curso_id,titulo_curso,cantidad_inscripciones
K0010,APIs con FastAPI,23
K0007,Modelado Dimensional,19
K0040,Prompt Engineering,18
K0028,Gestión de Proyectos,17
K0041,MLOps End to End,17
K0031,Liderazgo de Equipos,16
K0005,Estadística Práctica,16
K0024,Diseño de Interfaces,15
K0026,Accesibilidad Digital,15
K0034,Google Cloud Essentials,14


### 12. Inscripciones y tasa de finalización por categoría

Calcula por categoría la cantidad de inscripciones, completadas y la tasa de finalización.

In [0]:
%sql
WITH info_inscripciones AS (
    SELECT
        c.categoria,
        COUNT(1) AS inscripciones_totales,
        SUM(
            CASE
                WHEN i.estado = 'completado' THEN 1
                ELSE 0
            END
        ) AS inscripciones_completadas
        -- inscripciones_completadas / inscripciones_totales
    FROM cursos AS c
    LEFT JOIN inscripciones AS i
    USING (curso_id)
    GROUP BY c.categoria
)
SELECT 
    *,
    inscripciones_completadas / inscripciones_totales AS tasa_finalizacion
FROM info_inscripciones
ORDER BY tasa_finalizacion;

categoria,inscripciones_totales,inscripciones_completadas,tasa_finalizacion
Inteligencia Artificial,59,22,0.3728813559322034
Marketing,66,25,0.3787878787878788
Programación,90,37,0.4111111111111111
Negocios,72,30,0.4166666666666667
Data Analytics,99,44,0.4444444444444444
Cloud,70,34,0.4857142857142857
Diseño UX,67,38,0.5671641791044776


### 13. Ingresos aprobados por curso

Calcula los ingresos aprobados por curso.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(
        CASE
            WHEN p.estado_pago = 'aprobado' THEN p.monto
            ELSE 0
        END
    ) ingreso_total
FROM cursos AS c
LEFT JOIN inscripciones AS i
USING (curso_id)
LEFT JOIN pagos AS p
USING (inscripcion_id)
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id AND p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


In [0]:
%sql
--delimitanbdo universo 
WITH pagos_delimitado AS (
    SELECT *
    FROM pagos
    WHERE estado_pago = "aprobado"
)
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos_delimitado AS p
    ON i.inscripcion_id = p.inscripcion_id
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
WHERE p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
WHERE p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


### 14. Top 10 cursos por ingresos aprobados

Identifica los 10 cursos con mayores ingresos aprobados.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    SUM(p.monto) AS ingreso_total
FROM cursos AS c
INNER JOIN inscripciones AS i
    ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
    ON i.inscripcion_id = p.inscripcion_id
WHERE p.estado_pago = "aprobado"
GROUP BY c.curso_id, c.titulo_curso
ORDER BY ingreso_total DESC
LIMIT 10;

curso_id,titulo_curso,ingreso_total
K0037,Kubernetes Básico,1237.50
K0041,MLOps End to End,1116.25
K0026,Accesibilidad Digital,1092.25
K0010,APIs con FastAPI,1072.50
K0007,Modelado Dimensional,1056.25
K0005,Estadística Práctica,1039.50
K0012,POO con Python,1030.00
K0030,Productividad Personal,960.50
K0038,Infraestructura como Código,823.50
K0022,Figma desde Cero,798.00


### 15. Métricas por instructor

Muestra por instructor: cantidad de cursos, inscripciones totales e ingresos aprobados.

In [0]:
%sql
SELECT 
    nombre_instructor,
    COUNT(DISTINCT c.curso_id) AS cantidad_cursos,
    COUNT(DISTINCT i.inscripcion_id) AS inscripciones_totales,
    SUM(p.monto) AS ingresos_totales
FROM instructores AS ins
INNER JOIN cursos AS c
ON ins.instructor_id = c.instructor_id
INNER JOIN inscripciones AS i
ON c.curso_id = i.curso_id
INNER JOIN pagos AS p
ON i.inscripcion_id = p.inscripcion_id AND p.estado_pago ="aprobado"
GROUP BY ins.nombre_instructor

nombre_instructor,cantidad_cursos,inscripciones_totales,ingresos_totales
Rosa Paredes,2,25,549.00
Luis Silva,4,47,1856.00
Alejandra Ramos,2,19,922.75
Lucía Rojas,2,22,1116.25
José Paredes,3,30,1486.25
Andrea Mendoza,3,30,2211.00
Hugo Salazar,3,29,1460.25
Carlos Navarro,3,41,2714.50
Renzo Espinoza,1,12,340.50
Luis Cárdenas,2,27,1311.75


### 16. Estudiantes sin inscripciones

Lista los estudiantes que nunca se inscribieron en un curso.

In [0]:
%sql
SELECT
c.curso_id,
c.titulo_curso,
c.categoria
FROM cursos AS c
LEFT ANTI JOIN inscripciones AS i
ON c.curso_id = i.curso_id

curso_id,titulo_curso,categoria
K0043,Visión Computacional,Inteligencia Artificial
K0044,Agentes con IA,Inteligencia Artificial
K0045,Modelos Generativos,Inteligencia Artificial


### 17. Cursos sin inscripciones

Lista los cursos que nunca recibieron una inscripción.

In [0]:
%sql
SELECT
    c.curso_id,
    c.titulo_curso,
    c.categoria
FROM cursos AS c
LEFT ANTI JOIN inscripciones AS i
ON c.curso_id = i.curso_id

curso_id,titulo_curso,categoria
K0043,Visión Computacional,Inteligencia Artificial
K0044,Agentes con IA,Inteligencia Artificial
K0045,Modelos Generativos,Inteligencia Artificial


### 18. Nota promedio por nivel para cursos completados

Calcula la nota final promedio de los cursos completados, agrupada por nivel.

In [0]:
%sql
SELECT
    c.nivel,
    AVG(i.nota_final) AS nota_promedio
FROM inscripciones AS i
INNER JOIN cursos AS c
ON i.curso_id = c.curso_id AND i.estado = "completado"
GROUP BY c.nivel

nivel,nota_promedio
Intermedio,79.98
Avanzado,81.37777777777778
Básico,82.84705882352941


### 19. Minutos vistos por estudiante

Calcula el total de minutos vistos por estudiante y ordénalos de mayor a menor.

In [0]:
%sql
SELECT
    e.nombre_estudiante,
    SUM(COALESCE(s.minutos_vistos, 0)) AS total_minutos
FROM estudiantes AS e
LEFT JOIN inscripciones AS i
ON e.estudiante_id = i.estudiante_id
LEFT JOIN sesiones AS s
ON i.inscripcion_id = s.inscripcion_id
GROUP BY e.nombre_estudiante
ORDER BY total_minutos DESC

nombre_estudiante,total_minutos
Carlos Herrera,5893
Hugo Paredes,5553
Jorge Silva,5031
Diego Ramos,4701
Hugo Chávez,4461
José Herrera,4220
Alejandra Paredes,4176
Gabriela Rojas,4088
Lucía Chávez,4010
Hugo Núñez,3945


### 20. Calificación promedio y cantidad de reseñas por curso

Calcula por curso la calificación promedio y cantidad de reseñas, incluyendo cursos sin reseñas.

In [0]:
%sql
SELECT
    c.titulo_curso,
    AVG(i.nota_final) AS calificacion_promedio,
    COUNT(r.resena_id) AS cantidad_resenas
FROM cursos AS c
LEFT JOIN inscripciones AS i
ON i.curso_id = c.curso_id
LEFT JOIN resenas AS r
ON i.inscripcion_id = r.inscripcion_id
GROUP BY c.titulo_curso;

titulo_curso,calificacion_promedio,cantidad_resenas
Databricks Fundamentals,79.14285714285714,3
Angular Profesional,86.0,3
Growth Marketing,81.75,3
Excel para Analistas,77.33333333333333,4
POO con Python,82.0,5
Figma desde Cero,77.2,3
Agentes con IA,null,0
Git y GitHub,73.5,2
Research con Usuarios,81.8,4
Ventas Consultivas,84.66666666666667,1


### 21. Nuevos estudiantes por mes y acumulado

Calcula los nuevos estudiantes registrados por mes y el acumulado histórico.

In [0]:
%sql
WITH historico AS (
    SELECT
    DATE_TRUNC("MONTH",fecha_registro) AS mes,
    COUNT(1) AS numero_estudiantes
    FROM estudiantes
    GROUP BY DATE_TRUNC("MONTH",fecha_registro)
)
SELECT
mes,
numero_estudiantes,
SUM(numero_estudiantes) OVER(
ORDER BY mes
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW 
) AS acumulado
FROM historico
ORDER BY mes;

mes,numero_estudiantes,acumulado
2024-01-01T00:00:00.000Z,5,5
2024-02-01T00:00:00.000Z,3,8
2024-03-01T00:00:00.000Z,7,15
2024-04-01T00:00:00.000Z,9,24
2024-05-01T00:00:00.000Z,2,26
2024-06-01T00:00:00.000Z,8,34
2024-07-01T00:00:00.000Z,6,40
2024-08-01T00:00:00.000Z,6,46
2024-09-01T00:00:00.000Z,10,56
2024-10-01T00:00:00.000Z,6,62


### 22. Ingresos mensuales y crecimiento respecto al mes anterior

Calcula los ingresos aprobados por mes y el crecimiento porcentual respecto al mes anterior.

In [0]:
%sql
-- Escribe tu consulta aquí

### 23. Ranking de cursos por inscripciones dentro de cada categoría

Genera un ranking de cursos por cantidad de inscripciones dentro de cada categoría.

In [0]:
%sql
-- Escribe tu consulta aquí

### 24. Primera inscripción de cada estudiante

Obtén la primera inscripción de cada estudiante que tenga al menos una inscripción.

In [0]:
%sql
-- Escribe tu consulta aquí

### 25. Estudiantes inscritos en al menos 3 categorías

Lista los estudiantes inscritos en al menos 3 categorías distintas.

In [0]:
%sql
-- Escribe tu consulta aquí

### 26. Cursos con tasa de finalización superior a la global

Identifica los cursos cuya tasa de finalización es superior a la tasa de finalización global.

In [0]:
%sql
-- Escribe tu consulta aquí

### 27. Participación de cada instructor en los ingresos de su especialidad

Calcula el porcentaje de participación de cada instructor sobre los ingresos aprobados de su especialidad.

In [0]:
%sql
-- Escribe tu consulta aquí

### 28. Lifetime value y cuartil por estudiante

Calcula el valor de vida del estudiante (`lifetime_value`) usando pagos aprobados y clasifica a los estudiantes en 4 cuartiles con `NTILE(4)`.

In [0]:
%sql
-- Escribe tu consulta aquí

### 29. Porcentaje promedio de consumo por curso, limitado a 100% por inscripción

Calcula por curso el porcentaje promedio de consumo del contenido: minutos vistos divididos entre la duración del curso en minutos. Limita el resultado máximo de cada inscripción a 100%.

In [0]:
%sql
-- Escribe tu consulta aquí

### 30. Mayor pausa entre sesiones consecutivas por estudiante

Para cada estudiante, calcula la mayor cantidad de días de inactividad entre dos sesiones consecutivas. Muestra solo estudiantes cuya pausa máxima sea mayor a 14 días.

In [0]:
%sql
-- Escribe tu consulta aquí